In [ ]:
import visualizer

In [ ]:
import solvers.DWave_solver as DWave_solver
import map
import pathFormulation
import config.parser as config_parser
import QUBOBuilder
from config.hdf5parser import load_map_from_hdf5

In [ ]:
conf = config_parser.load_config("config/config.yaml", sections=["problems", "penalty_sets", "solver"])
map_hdf5 = load_map_from_hdf5("maps/synthetic/3x3/no_obs3x3_ter.h5")
grid = map.Grid.from_hdf5_data(map_hdf5)
map_conf = conf["problems"]["grid_3x3_default"]
problem = pathFormulation.PathfindingProblem.from_hdf5_data(map_hdf5, map_conf)

In [ ]:
material_costs = {
    "ceramic": 1.0,
    "water": 5.0,
    "asphalt": 1.5,
    "grass": 2.0
}
penalties_conf = conf["penalty_sets"]["ter"]
QUBOBuilder = QUBOBuilder.QUBOBuilder(problem, penalties=penalties_conf, name="standard", material_cost=material_costs)

In [ ]:
solver = DWave_solver.QUBOSolver(normalize_scale=2.0, num_reads=15)
QUBOBuilder.reset_problem()
QUBOBuilder.build()
solution = solver.solve_qubo(QUBOBuilder)
path = solver.decode_path(solution["solution"], problem)
print("Path:", path)
print(f"Energy: {solver.total_energy(solution):.4f}")
print(QUBOBuilder.get_num_wires())

### Visualization

In [ ]:
visualizer = visualizer.QuantumRoboticsVisualizer(
        grid_size=(grid.M, grid.N), 
        title="Pathfinding Visualization",
        start_image_path="images/scooby.svg", # e.g., "start.png" or "start.svg"
        goal_image_path="images/scoobysnack.svg"     # e.g., "goal.png" or "goal.svg"
    )

step_fig = visualizer.create_step_by_step_plot(obstacles=grid.obstacles, path=path, start=problem.start, goal=problem.end)
visualizer.show(step_fig)